### 1. Setup & Instalasi

In [1]:
import os
import json
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import MultiLabelBinarizer

print(f"TensorFlow version : {tf.__version__}")
print(f"NumPy version      : {np.__version__}")
print(f"Pandas version     : {pd.__version__}")
print("\nSetup selesai!")

TensorFlow version : 2.21.0
NumPy version      : 2.4.3
Pandas version     : 3.0.1

Setup selesai!


### 2.  Dataset Loader

In [ ]:
class DatasetLoader:
    """
    Memuat dan menyiapkan data dari file CSV
    yang nantinya disediakan oleh tim Data Scientist.

    File yang dibutuhkan (semua di folder data_dir):
    - fields.csv
    - roles.csv
    - master_roles.csv
    - learning_resources.csv
    - dummy_projects.csv
    - project_role_mapping.csv
    """

    def __init__(self, data_dir: str):
        # data_dir = path ke folder CSV, contoh: "data/"
        self.data_dir        = data_dir
        self.df_roles        = None
        self.df_fields       = None
        self.df_master       = None
        self.df_resources    = None
        self.df_projects     = None
        self.df_proj_mapping = None
        self.skill_vocab     = []
        self.mlb             = MultiLabelBinarizer()

    def load(self):
        """
        Muat semua file CSV dari data_dir.
        Dipanggil sekali saat FastAPI startup.
        """
        self.df_fields       = pd.read_csv(os.path.join(self.data_dir, "fields.csv"))
        self.df_roles        = pd.read_csv(os.path.join(self.data_dir, "roles.csv"))
        self.df_master       = pd.read_csv(os.path.join(self.data_dir, "master_roles.csv"))
        self.df_resources    = pd.read_csv(os.path.join(self.data_dir, "learning_resources.csv"))
        self.df_projects     = pd.read_csv(os.path.join(self.data_dir, "dummy_projects.csv"))
        self.df_proj_mapping = pd.read_csv(os.path.join(self.data_dir, "project_role_mapping.csv"))

        # Normalisasi nama kolom: lowercase dan hapus spasi
        for df in [self.df_roles, self.df_fields, self.df_master,
                   self.df_resources, self.df_projects, self.df_proj_mapping]:
            df.columns = [c.strip().lower() for c in df.columns]

        # Bangun skill vocabulary dari kolom skill di master_roles
        all_skills = []
        for skill_str in self.df_master["skill"].dropna():
            skills = [s.strip() for s in str(skill_str).split(",") if s.strip()]
            all_skills.extend(skills)

        self.skill_vocab = sorted(list(set(all_skills)))
        self.mlb.fit([self.skill_vocab])

        print(f"[DatasetLoader] Dataset berhasil dimuat dari: {self.data_dir}")
        print(f"[DatasetLoader] Total role  : {len(self.df_master)}")
        print(f"[DatasetLoader] Total skill : {len(self.skill_vocab)}")
        print(f"[DatasetLoader] Total field : {len(self.df_fields)}")
        return self

    def get_role_skill_matrix(self):
        """
        Buat matrix skill per role — shape: (n_roles, n_skills)
        Nilai 1 = role butuh skill, 0 = tidak.
        Dipakai sebagai role profile di Pipeline 2.
        """
        role_skills = []
        for _, row in self.df_master.iterrows():
            if pd.notna(row.get("skill")):
                skills = [s.strip() for s in str(row["skill"]).split(",") if s.strip()]
            else:
                skills = []
            role_skills.append(skills)
        return self.mlb.transform(role_skills).astype(np.float32)

    def get_role_riasec_matrix(self):
        """
        Buat matrix RIASEC per role — shape: (n_roles, 6)
        Kolom: R, I, A, S, E, C — nilai 0-1 (dinormalisasi tim DS).
        Dipakai sebagai role profile di Pipeline 3.
        """
        riasec_cols = ["r", "i", "a", "s", "e", "c"]
        return self.df_roles[riasec_cols].fillna(0).values.astype(np.float32)

    def get_roadmap_by_role(self, role_id: str):
        """
        Ambil roadmap dari dataset berdasarkan role_id.
        Return: learning_path (step by step) + dummy_projects.
        Tidak menggunakan Generative AI — semua dari dataset DS.
        """
        # Learning path dari learning_resources
        resources = self.df_resources[
            self.df_resources["role_id"] == role_id
        ].sort_values("step_number")

        learning_path = [
            {
                "step"       : int(row["step_number"]),
                "nama_skill" : row["nama_skill"],
                "link_course": row["link_course"],
                "tipe"       : row["tipe"],
                "platform"   : row["platform"]
            }
            for _, row in resources.iterrows()
        ]

        # Dummy projects dari project_role_mapping
        proj_ids = self.df_proj_mapping[
            self.df_proj_mapping["role_id"] == role_id
        ]["project_id"].tolist()

        projects = self.df_projects[self.df_projects["project_id"].isin(proj_ids)]

        dummy_projects = [
            {
                "judul"      : row["judul_project"],
                "brief_case" : row["brief_case"],
                "tools_used" : row["tools_used"]
            }
            for _, row in projects.iterrows()
        ]

        return {"learning_path": learning_path, "dummy_projects": dummy_projects}

### Inisialisasi Dataset Loader

In [ ]:
DATA_DIR = "data/"

loader = DatasetLoader(data_dir=DATA_DIR)
loader.load()

In [ ]:
# Cek struktur data
print("=== master_roles ===")
print(loader.df_master.head(3))
print()
print("=== roles (RIASEC) ===")
print(loader.df_roles.head(3))

### 3. Helper Functions

Fungsi pendukung untuk Pipeline 3 (RIASEC).

In [ ]:
def get_interest_code(riasec_scores: dict) -> str:
    """
    Ambil 3 huruf RIASEC dengan skor tertinggi sebagai Interest Code.
    Ditampilkan di halaman hasil tes minat bakat.

    Contoh:
      Input : {"r": 0.8, "i": 0.9, "a": 0.4, "s": 0.6, "e": 0.5, "c": 0.7}
      Output: "I R C"  (karena I=0.9 > R=0.8 > C=0.7)
    """
    sorted_riasec = sorted(riasec_scores.items(), key=lambda x: x[1], reverse=True)
    top3 = [k.upper() for k, _ in sorted_riasec[:3]]
    return " ".join(top3)


def get_sector_recommendations(riasec_scores: dict, loader: DatasetLoader,
                                riasec_model: tf.keras.Model) -> list:
    """
    Hitung persentase relevansi tiap sektor industri berdasarkan
    rata-rata skor role per bidang.
    Ditampilkan sebagai 'Rekomendasi Sektor Industri yang Relevan'.

    Return: list dict diurutkan dari skor tertinggi
    [
      {"field_id": "F02", "field_name": "Data Science & AI", "relevance_pct": 91.0},
      ...
    ]
    """
    role_riasec_matrix = loader.get_role_riasec_matrix()  # (n_roles, 6)

    # Susun vektor RIASEC user
    user_vector = np.array([[
        riasec_scores.get("r", 0), riasec_scores.get("i", 0),
        riasec_scores.get("a", 0), riasec_scores.get("s", 0),
        riasec_scores.get("e", 0), riasec_scores.get("c", 0),
    ]], dtype=np.float32)

    # Batch inference — hitung semua role sekaligus
    n_roles     = len(role_riasec_matrix)
    user_matrix = np.repeat(user_vector, n_roles, axis=0)
    all_scores  = riasec_model.predict(
        [user_matrix, role_riasec_matrix], verbose=0
    ).flatten().tolist()

    field_names = {
        "F01": "Teknologi Informasi & Software Development",
        "F02": "Data Science & Artificial Intelligence",
        "F03": "Desain Kreatif & UI/UX",
        "F04": "Digital Marketing & Analytics"
    }

    # Kelompokkan skor per bidang
    sector_scores = {fid: [] for fid in field_names}
    for idx, row in loader.df_master.iterrows():
        role_id = row["role_id"]
        field_id_match = loader.df_roles[
            loader.df_roles["role_id"] == role_id
        ]["field_id"].values
        if len(field_id_match) > 0:
            fid = field_id_match[0]
            if fid in sector_scores:
                sector_scores[fid].append(all_scores[idx])

    # Hitung rata-rata per sektor
    result = []
    for fid, scores in sector_scores.items():
        if scores:
            result.append({
                "field_id"     : fid,
                "field_name"   : field_names[fid],
                "relevance_pct": round(sum(scores) / len(scores), 1)
            })

    return sorted(result, key=lambda x: x["relevance_pct"], reverse=True)

### Test Helper Functions

In [ ]:
# Contoh penggunaan get_interest_code
sample_riasec = {"r": 0.8, "i": 0.9, "a": 0.4, "s": 0.6, "e": 0.5, "c": 0.7}
interest_code = get_interest_code(sample_riasec)
print(f"Interest Code: {interest_code}")
# Output yang diharapkan: "I R C"

### 4. Custom Components

Custom Layer 1 - SkillEmbeddingLayer

In [ ]:
class SkillEmbeddingLayer(tf.keras.layers.Layer):
    """
    Custom Layer 
    Proyeksikan multi-hot skill vector ke dense embedding space.
    Mengubah representasi 0/1 skill menjadi representasi bermakna.
    Dipakai di Pipeline 2 (skill matching).

    Input  : multi-hot vector ukuran (n_skills,)
    Proses : Dense(embedding_dim) → LayerNorm → GeLU activation
    Output : dense embedding ukuran (embedding_dim,)
    """
    def __init__(self, embedding_dim=128, **kwargs):
        super().__init__(**kwargs)
        self.embedding_dim = embedding_dim
        self.dense = tf.keras.layers.Dense(embedding_dim)
        self.norm  = tf.keras.layers.LayerNormalization()

    def call(self, inputs, training=False):
        x = self.dense(inputs)
        x = self.norm(x, training=training)
        return tf.nn.gelu(x)     # GeLU lebih smooth dari ReLU

    def get_config(self):
        cfg = super().get_config()
        cfg.update({"embedding_dim": self.embedding_dim})
        return cfg

print("SkillEmbeddingLayer berhasil didefinisikan")"

Custom Layer 2 - CosineSimilarityLayer

In [ ]:
class CosineSimilarityLayer(tf.keras.layers.Layer):
    """
    Custom Layer 
    Hitung cosine similarity antara user embedding dan role embedding.
    Dipakai di Pipeline 2 dan Pipeline 3.

    Input  : [user_emb, role_emb] — dua tensor ukuran sama
    Proses : L2 normalize keduanya → dot product → skala ke [0, 100]
    Output : similarity score dalam persentase 0-100
    """
    def call(self, inputs):
        user_emb, role_emb = inputs
        user_norm  = tf.nn.l2_normalize(user_emb, axis=-1)
        role_norm  = tf.nn.l2_normalize(role_emb, axis=-1)
        similarity = tf.reduce_sum(user_norm * role_norm, axis=-1, keepdims=True)
        # Konversi dari rentang [-1, 1] ke [0, 100]
        return (similarity + 1) / 2 * 100

    def get_config(self):
        return super().get_config()

print("CosineSimilarityLayer berhasil didefinisikan")"

Custom Loss - WeightedMatchingLoss

In [ ]:
class WeightedMatchingLoss(tf.keras.losses.Loss):
    """
    Custom Loss 
    Weighted Binary Cross Entropy untuk menangani class imbalance.
    Role yang jarang muncul di dataset diberi bobot lebih tinggi
    agar model tetap belajar mengenalinya.

    role_weights : array bobot per role, shape (n_roles,)
                   None = semua role diberi bobot sama
    """
    def __init__(self, role_weights=None, **kwargs):
        super().__init__(**kwargs)
        self.role_weights = role_weights

    def call(self, y_true, y_pred):
        bce = tf.keras.losses.binary_crossentropy(y_true, y_pred)
        if self.role_weights is not None:
            weights = tf.cast(self.role_weights, dtype=tf.float32)
            bce     = bce * weights
        return tf.reduce_mean(bce)

    def get_config(self):
        cfg = super().get_config()
        cfg.update({"role_weights": self.role_weights})
        return cfg

print("WeightedMatchingLoss berhasil didefinisikan")

Custom Callback - CareerLensCallback

In [ ]:
class CareerLensCallback(tf.keras.callbacks.Callback):
    """
    Custom Callback 
    Dipanggil otomatis setiap akhir epoch untuk:
    - Monitor similarity score rata-rata di validation set
    - Log metrik ke TensorBoard (val_avg_similarity, loss)
    - Simpan model terbaik ke saved_model/careerlens_best.keras
    - Early stopping jika skor tidak membaik selama `patience` epoch
    """
    def __init__(self, val_data, log_dir="logs/", patience=5):
        super().__init__()
        self.val_data   = val_data
        self.patience   = patience
        self.best_score = -999
        self.wait       = 0
        self.writer     = tf.summary.create_file_writer(log_dir)

    def on_epoch_end(self, epoch, logs=None):
        X_val, y_val = self.val_data
        preds        = self.model.predict(X_val, verbose=0)

        # Handle output dict (SkillModel) vs tensor (RIASECModel)
        if isinstance(preds, dict):
            pred_scores = preds.get("role_score", preds)
        else:
            pred_scores = preds

        avg_score = float(np.mean(pred_scores))

        # Log ke TensorBoard
        with self.writer.as_default():
            tf.summary.scalar("val_avg_similarity", avg_score, step=epoch)
            if logs:
                for key, val in logs.items():
                    tf.summary.scalar(key, val, step=epoch)

        print(f"\n[CareerLensCallback] Epoch {epoch+1} | Avg Similarity: {avg_score:.2f}%")

        # Simpan model terbaik + early stopping
        if avg_score > self.best_score:
            self.best_score = avg_score
            self.wait       = 0
            os.makedirs("saved_model", exist_ok=True)
            self.model.save("saved_model/careerlens_best.keras")
            print(f" Model tersimpan — best score: {self.best_score:.2f}%")
        else:
            self.wait += 1
            if self.wait >= self.patience:
                print(f" Early stopping di epoch {epoch+1}")
                self.model.stop_training = True

print("CareerLensCallback berhasil didefinisikan")"